# Part 2, Question 2: OLAP Queries (PostgreSQL)

This notebook demonstrates OLAP operations using PostgreSQL queries on the `staging_sales` table (or the star schema created in Question 1). For simplicity, these queries will run directly against the `staging_sales` table.

---

### a) Slicing

**Business Question:** What were the total sales for the 'Food and beverages' product line?

**SQL Implementation:**
```sql
SELECT
    SUM(total) AS total_sales
FROM
    staging_sales
WHERE
    product_line = 'Food and beverages';

-- Business Answer: The query result shows the sum of all sales for the 'Food and beverages' product line, providing a specific slice of the overall sales data.
b) DicingBusiness Question: What were the sales to female 'Member' customers in the city of 'Yangon'?SQL Implementation:SELECT
    COUNT(invoice_id) AS number_of_transactions,
    SUM(total) AS total_sales
FROM
    staging_sales
WHERE
    city = 'Yangon'
    AND customer_type = 'Member'
    AND gender = 'Female';

-- Business Answer: This query dices the data cube by filtering on three dimensions (city, customer type, and gender) to provide a very specific sub-view of performance.
c) PivotingBusiness Question: How do total sales compare across different product lines and payment methods?SQL Implementation:Pivoting in PostgreSQL is typically done using the crosstab function, which requires the tablefunc extension.-- First, ensure the extension is enabled.
CREATE EXTENSION IF NOT EXISTS tablefunc;

-- The query for crosstab needs a source query that returns three columns:
-- 1. Row identifier (product_line)
-- 2. Category to become new columns (payment_method)
-- 3. Value to be placed in the new columns (total sales)
SELECT *
FROM crosstab(
  'SELECT
     product_line,
     payment_method,
     SUM(total)
   FROM staging_sales
   GROUP BY product_line, payment_method
   ORDER BY product_line, payment_method',
  'SELECT DISTINCT payment_method FROM staging_sales ORDER BY 1'
) AS ct (
  product_line VARCHAR,
  "Cash" NUMERIC,
  "Credit Card" NUMERIC,
  "Ewallet" NUMERIC
);

-- Business Answer: The resulting table clearly displays sales for each product line broken down by payment method, making it easy to compare them side-by-side.
d) Drill DownBusiness Question: We know the total sales for each city. What is the sales breakdown by branch within the city of 'Mandalay'?SQL Implementation:-- 1. High-level view (Sales by City)
SELECT
    city,
    SUM(total) AS total_sales
FROM
    staging_sales
GROUP BY
    city
ORDER BY
    total_sales DESC;

-- 2. Drill Down into 'Mandalay'
SELECT
    branch,
    SUM(total) AS total_sales
FROM
    staging_sales
WHERE
    city = 'Mandalay'
GROUP BY
    branch;

-- Business Answer: After seeing the aggregate sales for Mandalay, the second query drills down to reveal the performance of the individual branch within that city.
e) Drill-Up (Roll-Up)Business Question: We have daily sales data. What were the total sales for each month, with a grand total for the year?SQL Implementation:The ROLLUP function is perfect for this. It aggregates data at multiple levels in a single query.SELECT
    EXTRACT(YEAR FROM sale_date) AS sales_year,
    EXTRACT(MONTH FROM sale_date) AS sales_month,
    SUM(total) as total_sales
FROM
    staging_sales
GROUP BY
    ROLLUP(sales_year, sales_month)
ORDER BY
    sales_year, sales_month;

-- Business Answer: This query rolls up daily sales data to provide monthly totals. The row where 'sales_month' is NULL represents the rolled-up total for the entire year.
f) Drill AcrossBusiness Question: How does the total quantity of items sold correlate with the average customer rating for each product line?SQL Implementation:This is achieved by joining two separate aggregated queries on their common dimension.WITH quantity_summary AS (
    SELECT
        product_line,
        SUM(quantity) AS total_quantity_sold
    FROM staging_sales
    GROUP BY product_line
),
rating_summary AS (
    SELECT
        product_line,
        AVG(rating) AS average_rating
    FROM staging_sales
    GROUP BY product_line
)
SELECT
    qs.product_line,
    qs.total_quantity_sold,
    rs.average_rating
FROM
    quantity_summary qs
JOIN
    rating_summary rs ON qs.product_line = rs.product_line
ORDER BY
    qs.total_quantity_sold DESC;

-- Business Answer: By drilling across two different business processes (sales volume and customer satisfaction), we can analyze their relationship for each product line.
g) Drill ThroughBusiness Question: The summary shows total sales over $55,000 for the 'Fashion accessories' product line. Can we see the actual individual transactions that contribute to this total?SQL Implementation:-- 1. The summary query

SELECT
    SUM(total) AS total_fashion_sales
FROM
    staging_sales
WHERE
    product_line = 'Fashion accessories';


-- 2. Drill through to the transactional data
SELECT
    invoice_id,
    sale_date,
    quantity,
    unit_price,
    total,
    rating
FROM
    staging_sales
WHERE
    product_line = 'Fashion accessories'
ORDER BY
    sale_date;

-- Business Answer: After viewing an aggregated number, this query drills all the way through to the raw transaction logs, allowing for a detailed audit or inspection of the underlying data.